In [ ]:
import glob
import os

import ehtim as eh
from astropy.io import fits
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from eht_inspection.uvfits import *
from eht_inspection.utils import scan_ids_from_intervals
from eht_inspection.plotting import plot_uv_coverage


In [ ]:
uvfits_files = sorted(glob.glob("**/*.uvfits", recursive=True))

print(f"Found {len(uvfits_files)} uvfits files:")
for i, f in enumerate(uvfits_files):
    print(i, f)

fname = uvfits_files[0]
print("\nUsing:", fname)

print("\nFITS structure:")
with fits.open(fname) as hdul:
    hdul.info()

print("\nLoading with eht-inspection...")
uvdata = load_obs_uvfits(fname, IF=all, return_dict=True)
times = uvdata["times"]
t1 = uvdata["t1"]
t2 = uvdata["t2"]
u = uvdata["u"]
v = uvdata["v"]
rr = uvdata["rr"]
rl = uvdata["rl"]
lr = uvdata["lr"]
ll = uvdata["ll"]
rrsigma = uvdata["rrsigma"]
rlsigma = uvdata["rlsigma"]
lrsigma = uvdata["lrsigma"]
llsigma = uvdata["llsigma"]

obs = eh.obsdata.load_uvfits(fname, IF=[0], polrep='circ', remove_nan=True)
obs.add_scans()
scans = obs.scans
scan_ids = scan_ids_from_intervals(times, scans)

print(obs)
print("\nData fields:")
print(obs.data.dtype.names)

print("\nSites:")
print(obs.tarr["site"])

print("\nNumber of visibilities:", len(obs.data))

In [ ]:
fig, ax = plot_uv_coverage(u, v)
plt.show()


For each baseline, for all 4 polarizations (in a single plot), show amp vs. channel, phase vs. channel for raw data

amp vs. time/scan/uv distance, phase vs. time/scan for channel-averaged, 10s-averaged data.

Raw data can tell you bandpass, bad channels, cross-pol channel structure

Averaged data can tell you scan-to-scan jumps, amplitude stability, and time-domain outliers

In [ ]:
print(np.unique(scan_ids))

In [ ]:
# scannum must be in the scan_ids
scannum = np.unique(scan_ids)[0]
result = build_scan_coherency_matrix_from_uvfits(fname, scannum, scans=scans, IF=all)

### For Raw Data

The repo only has channel-averaged and 10s-averaged data due to size limit (~140MB)

In [ ]:
fig, axs = plot_scan_phase_vs_channel_all_baselines(
    allcoh=result["allcoh"],
    channel_list=result["channel_list"],
    station_list=result["station_list"],
    scan_num=result["scan_number"],
    average_over_time=True,
    unwrap_phase=False,
    source="3C273",
)

In [ ]:
fig, axs = plot_scan_amp_vs_channel_all_baselines(
    allcoh=result["allcoh"],
    channel_list=result["channel_list"],
    station_list=result["station_list"],
    scan_num=result["scan_number"],
    average_over_time=True,
    source="3C273",
)

In [ ]:
# Make one plot per scan
for sscan in np.unique(scan_ids):
    tmp_res = build_scan_coherency_matrix(sscan, scan_ids, times, t1, t2, u, v, rr, rl, lr, ll)

    fig, axs = plot_scan_phase_vs_channel_all_baselines(
        allcoh=tmp_res["allcoh"],
        channel_list=tmp_res["channel_list"],
        station_list=tmp_res["station_list"],
        scan_num=tmp_res["scan_number"],
        average_over_time=True,
        unwrap_phase=False,
        source="3C273",
    )

    plt.close(fig)
    del fig, axs

    fig, axs = plot_scan_amp_vs_channel_all_baselines(
        allcoh=tmp_res["allcoh"],
        channel_list=tmp_res["channel_list"],
        station_list=tmp_res["station_list"],
        scan_num=tmp_res["scan_number"],
        average_over_time=True,
        source="3C273",
    )

    plt.close(fig)
    del tmp_res
    del fig, axs
    
    

### For Averaged Data

In [ ]:
fig, ax = plot_result_phase_vs_time_all_baselines(result, source="3C273")

In [ ]:
fig, ax = plot_result_amp_vs_time_all_baselines(result, source="3C273")

In [ ]:
results = []

for sscan in np.unique(scan_ids):
    res = build_scan_coherency_matrix(
        sscan,
        scan_ids,
        times,
        t1,
        t2,
        u,
        v,
        rr,
        rl,
        lr,
        ll,
    )
    results.append(res)

In [ ]:
fig, axs, out = plot_results_amp_vs_scan_all_baselines(results, source="3C273")

In [ ]:
fig, axs, out = plot_results_phase_vs_scan_all_baselines(results, source="3C273")

In [ ]:
fig, axs = plot_amp_uvdist_whole_dataset(u, v, rr, rl, lr, ll, source="3C273")

In [ ]:
for tmp_res in results:
    fig, ax = plot_result_phase_vs_time_all_baselines(tmp_res)
    plt.close()
    del fig, ax

    fig, ax = plot_result_amp_vs_time_all_baselines(tmp_res)
    plt.close()
    del fig, ax

### Inspect Closure Quantities 

In [ ]:
result = build_scan_coherency_matrix_from_uvfits(fname, 7, scans=scans, IF=all)

In [ ]:
closure = build_closure_products_from_coherency(
    result,
    exclude_stations=("AA", "ALMA"),
)

In [ ]:
fig, ax = plot_closure_phase_vs_time_all_triangles(closure, source="3C273")

In [ ]:
fig, ax = plot_closure_amp_vs_time_all_quadrangles(closure, source="3C273")